<a href="https://colab.research.google.com/github/abeeraz379/Logistic-Regression-Optimization-Stroke-data-/blob/main/stroke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# import libraries

In [53]:
# import numpy ,pandas
# import test train split
# import logisticregression
# import Gridsearchcv
# import scaler
# import pipeline
# import onehot encoder
# import simple imbuter
# import confusion matrix , confusion report
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix,classification_report
import warnings
warnings.filterwarnings('ignore')

# load and inspect data

In [2]:
# load stroke data
df=pd.read_csv("/content/drive/MyDrive/AXSOSACADEMY/AXSOSACADEMY/02-IntroML/Week07/Data/stroke.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,1192,Female,31,0,0,No,Govt_job,Rural,70.66,27.2,never smoked,0
1,77,Female,13,0,0,No,children,Rural,85.81,18.6,Unknown,0
2,59200,Male,18,0,0,No,Private,Urban,60.56,33.0,never smoked,0
3,24905,Female,65,0,0,Yes,Private,Urban,205.77,46.0,formerly smoked,1
4,24257,Male,4,0,0,No,children,Rural,90.42,16.2,Unknown,0


In [3]:
# show info about df
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1137 entries, 0 to 1136
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1137 non-null   int64  
 1   gender             1137 non-null   object 
 2   age                1137 non-null   object 
 3   hypertension       1137 non-null   int64  
 4   heart_disease      1137 non-null   int64  
 5   ever_married       1137 non-null   object 
 6   work_type          1137 non-null   object 
 7   Residence_type     1137 non-null   object 
 8   avg_glucose_level  1137 non-null   float64
 9   bmi                1085 non-null   float64
 10  smoking_status     1137 non-null   object 
 11  stroke             1137 non-null   int64  
dtypes: float64(2), int64(4), object(6)
memory usage: 106.7+ KB


In [4]:
# show statistical info about df
df.describe()

,id,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,1137.000000,1137.000000,1137.000000,1137.000000,1085.000000,1137.000000
mean,36750.933157,0.118734,0.068602,107.664002,29.198065,0.120493
std,21112.281253,0.323617,0.252887,47.618723,7.669615,0.325680
min,77.000000,0.000000,0.000000,55.270000,11.300000,0.000000
25%,17986.000000,0.000000,0.000000,77.600000,24.100000,0.000000
50%,37479.000000,0.000000,0.000000,91.820000,28.500000,0.000000
75%,55410.000000,0.000000,0.000000,113.850000,33.200000,0.000000
max,72918.000000,1.000000,1.000000,266.590000,64.400000,1.000000


In [5]:
# show statistical info about object features
df.describe(include="object")

,gender,age,ever_married,work_type,Residence_type,smoking_status
count,1137,1137,1137,1137,1137,1137
unique,3,84,2,5,2,4
top,Female,79,Yes,Private,Urban,never smoked
freq,642,26,769,672,587,416


In [6]:
# show values of each column
for col in df.columns:
  print(col)
  print(df[col].unique())

id
[ 1192    77 59200 ... 59749 12109  5731]
gender
['Female' 'Male' 'Other']
age
['31' '13' '18' '65' '4' '28' '64' '62' '26' '63' '50' '77' '1' '52' '24'
 '48' '45' '74' '3' '17' '54' '55' '67' '35' '38' '81' '34' '44' '79' '56'
 '70' '30' '39' '29' '80' '59' '51' '19' '43' '71' '0' '23' '53' '78' '66'
 '60' '76' '22' '6' '47' '2' '46' '11' '33' '37' '49' '61' '75' '40' '8'
 '57' '12' '21' '41' '58' '27' '20' '68' '42' '5' '69' '9' '36' '25' '82'
 '73' '32' '7' '16' '14' '72' '15' '10' '*82']
hypertension
[0 1]
heart_disease
[0 1]
ever_married
['No' 'Yes']
work_type
['Govt_job' 'children' 'Private' 'Self-employed' 'Never_worked']
Residence_type
['Rural' 'Urban']
avg_glucose_level
[ 70.66  85.81  60.56 ... 234.35  80.43 108.61]
bmi
[27.2 18.6 33.  46.  16.2 30.3 31.4 31.7 20.3 27.4 34.5 32.8 17.2 32.3
 26.1 32.2 29.4 26.8 21.7 18.3 22.9 20.8 27.  25.6 24.8 27.6 44.6 43.2
 33.2 30.7 37.6 35.5 25.2 31.1 36.  21.4 24.2 26.3 29.  28.4 37.8 30.2
 40.1 45.9 33.4 31.2 29.1 27.3 24.6 23.6 20.

# Clean Data

In [7]:
# check missing values
df.isna().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,52


In [8]:
# check duplicates
df.duplicated().sum()

np.int64(0)

# Preprocessing

In [9]:
# define x and y
x=df.drop(columns="stroke")
y=df["stroke"]

In [10]:
y.value_counts()

,count
stroke,
0,1000
1,137


In [11]:
# split data
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=1)

In [12]:
# impute missing value with median
# build scaler
# build one hot encoder

num_cols = x_train.select_dtypes("number").columns
cat_cols = x_train.select_dtypes("object").columns
from sklearn.compose import ColumnTransformer
num_transformer = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
cat_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))
preprocessor = ColumnTransformer([('num', num_transformer, num_cols), ('cat', cat_transformer, cat_cols)])


# Build Model

In [13]:
# build logistic regression moddel
lr=LogisticRegression(random_state=1)

In [14]:
# make pipeline
pipe=make_pipeline(preprocessor,lr)


In [15]:
# fir pipe on data
pipe.fit(x_train,y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['id', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'age', 'ever_married', 'work_type', 'Residence_type',
       'smoking_status'],
      dtype='object'))])),
                ('logisticregression', LogisticRegression(random_state=1))])

# Evaluate

In [16]:
# evaluate model using confusion matrix and report
# print accuracy of the model
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))


[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285

accuracy Training: 0.8838028169014085
accuracy Testing: 0.8666666666666667


# Tuning to Get better results

In [17]:
# tunning the model using gridsearchcv
pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('num',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     StandardScaler())]),
                                    Index(['id', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'], dtype='object')),
                                   ('cat',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehotencoder',
                                                     OneHotEncoder(handle_unknown='ignore'))]),
                                    Index(['gender', 'age', 'ever_married', 'work_type', 'Residence_t

# Change Values of Parameters to get the best_params_

## C

In [18]:
# try diffrent values of 'logisticregression__C' to get the best_params_
param_grid={"logisticregression__C":[0.001,0.01,0.1, 1, 10, 100, 1000]}
grid=GridSearchCV(pipe,param_grid,cv=5)
grid.fit(x_train,y_train)
# get the best value
grid.best_params_

{'logisticregression__C': 0.001}

In [19]:
# rewrite the model with best value for c
lr=LogisticRegression(random_state=1,C=0.001)
pipe=make_pipeline(preprocessor,lr)
# evaluate the new version of model
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))


[[250   0]
 [ 35   0]]
              precision    recall  f1-score   support

           0       0.88      1.00      0.93       250
           1       0.00      0.00      0.00        35

    accuracy                           0.88       285
   macro avg       0.44      0.50      0.47       285
weighted avg       0.77      0.88      0.82       285

accuracy Training: 0.8802816901408451
accuracy Testing: 0.8771929824561403


**The F1-Score increase alittle bit, but the results for class 1 are all zeros**

## solver

In [20]:
# # try diffrent values of 'logisticregression__solver' to get the best_params_
param_grid={"logisticregression__solver":["newton-cg", "lbfgs", "liblinear", "sag", "saga"]}
grid=GridSearchCV(pipe,param_grid,cv=5)
grid.fit(x_train,y_train)
# get the best value
grid.best_params_


{'logisticregression__solver': 'newton-cg'}

In [21]:

# build the new version with newton-cg
lr=LogisticRegression(random_state=1,solver="newton-cg")
# evaluate the model
pipe=make_pipeline(preprocessor,lr)
# evaluate the new version of model
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))


[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285

accuracy Training: 0.8838028169014085
accuracy Testing: 0.8666666666666667


**The reults is the same for the defult parameters**

##penalty

In [22]:
# try diffrent values of logisticregression__penalty and 'logisticregression__solver' to get the best_params_
param_grid = {
    'logisticregression__penalty': ['l1', 'l2'],
    'logisticregression__solver': ['liblinear', 'saga']
}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)

{'logisticregression__penalty': 'l2', 'logisticregression__solver': 'saga'}


In [23]:
# try the new values of parameters
lr = LogisticRegression(random_state=1, penalty='l1', solver='liblinear')
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[243   7]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.97      0.93       250
           1       0.30      0.09      0.13        35

    accuracy                           0.86       285
   macro avg       0.59      0.53      0.53       285
weighted avg       0.81      0.86      0.83       285



**the F1-Score Decrease**

In [24]:
# try the another values of parameters
lr = LogisticRegression(random_state=1, penalty='l2')
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285



## max_iter

In [25]:
# Change the max_iter
param_grid = {'logisticregression__max_iter': [10,20,50,100,200,300,400,500]}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)


{'logisticregression__max_iter': 10}


In [26]:
# try the new value of parameters
lr = LogisticRegression(random_state=1, max_iter=10)
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285



**The results didn't change, logisticregression__max_iter is Not The Most effect Parameter**

## class_weight

In [27]:
 # try diffrent values of logisticregression__class_weight to get the best_params_
param_grid = {'logisticregression__class_weight': ['balanced', None]}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)



{'logisticregression__class_weight': None}


In [28]:
# try the new value of parameters
lr = LogisticRegression(random_state=1,class_weight='balanced')
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[193  57]
 [ 14  21]]
              precision    recall  f1-score   support

           0       0.93      0.77      0.84       250
           1       0.27      0.60      0.37        35

    accuracy                           0.75       285
   macro avg       0.60      0.69      0.61       285
weighted avg       0.85      0.75      0.79       285



**The weighted avg increases, However the F1-Score decreaes**

## l1_ratio

In [29]:
 # try diffrent values of logisticregression__l1_ratio to get the best_params_
param_grid = {'logisticregression__l1_ratio': [0.0, 0.25, 0.5, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)

{'logisticregression__l1_ratio': 0.0}


In [30]:
# try the new value
lr = LogisticRegression(random_state=1,l1_ratio=0.0)
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285



**it Didn't Change**

# Result

**For this Data Set - > The moset effective parameter is c , then pantly and class_weight**

In [31]:
# Build The model
lr=LogisticRegression(random_state=1, penalty='l2',solver="newton-cg",C=0.001)
pipe=make_pipeline(preprocessor,lr)
# evaluate the new version of model
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))

[[250   0]
 [ 35   0]]
              precision    recall  f1-score   support

           0       0.88      1.00      0.93       250
           1       0.00      0.00      0.00        35

    accuracy                           0.88       285
   macro avg       0.44      0.50      0.47       285
weighted avg       0.77      0.88      0.82       285

accuracy Training: 0.8802816901408451
accuracy Testing: 0.8771929824561403


In [32]:
param_grid = {'logisticregression__l1_ratio': [0.0, 0.25, 0.5, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1],
              'logisticregression__class_weight': ['balanced', None],
              'logisticregression__max_iter': [10,20,50,100,200,300,400,500],
               'logisticregression__penalty': ['l1', 'l2'],
               'logisticregression__solver': ['liblinear', 'saga'],
              "logisticregression__C":[0.001,0.01,0.1, 1, 10, 100, 1000]
}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)

{'logisticregression__C': 0.01, 'logisticregression__class_weight': None, 'logisticregression__l1_ratio': 0.0, 'logisticregression__max_iter': 10, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'liblinear'}


In [33]:
# Build The model
lr=LogisticRegression(random_state=1, penalty='l2',solver="liblinear",C=0.01)
pipe=make_pipeline(preprocessor,lr)
# evaluate the new version of model
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))

[[250   0]
 [ 35   0]]
              precision    recall  f1-score   support

           0       0.88      1.00      0.93       250
           1       0.00      0.00      0.00        35

    accuracy                           0.88       285
   macro avg       0.44      0.50      0.47       285
weighted avg       0.77      0.88      0.82       285

accuracy Training: 0.8826291079812206
accuracy Testing: 0.8771929824561403


# Balancing

## Under sampling


In [34]:
# Under sampling to make the data balance
from imblearn.under_sampling import RandomUnderSampler
rus=RandomUnderSampler(random_state=1)
under_sample_pipe=make_pipeline(preprocessor,rus)
x_train_rus,y_train_rus=rus.fit_resample(x_train,y_train)
y_train_rus.value_counts()


,count
stroke,
0,102
1,102


In [66]:
# Build The model after undersampling with best parameter
#import imblearn pipeline
from imblearn.pipeline import make_pipeline as make_pipeline_imb
lr=LogisticRegression(random_state=1, penalty='l2',solver="newton-cg",C=0.01)
under_sample_pipe=make_pipeline_imb(preprocessor,rus,lr)
under_sample_pipe.fit(x_train_rus,y_train_rus)
y_pred=under_sample_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",under_sample_pipe.score(x_train_rus,y_train_rus))
print("accuracy Testing:",under_sample_pipe.score(x_test,y_test))



[[187  63]
 [ 18  17]]
              precision    recall  f1-score   support

           0       0.91      0.75      0.82       250
           1       0.21      0.49      0.30        35

    accuracy                           0.72       285
   macro avg       0.56      0.62      0.56       285
weighted avg       0.83      0.72      0.76       285

accuracy Training: 0.6666666666666666
accuracy Testing: 0.7157894736842105


In [67]:
# Build The model
lr=LogisticRegression(random_state=1)
under_sample_pipe=make_pipeline_imb(preprocessor,rus,lr)
under_sample_pipe.fit(x_train_rus,y_train_rus)
y_pred=under_sample_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",under_sample_pipe.score(x_train_rus,y_train_rus))
print("accuracy Testing:",under_sample_pipe.score(x_test,y_test))

[[169  81]
 [ 15  20]]
              precision    recall  f1-score   support

           0       0.92      0.68      0.78       250
           1       0.20      0.57      0.29        35

    accuracy                           0.66       285
   macro avg       0.56      0.62      0.54       285
weighted avg       0.83      0.66      0.72       285

accuracy Training: 0.7990196078431373
accuracy Testing: 0.6631578947368421


## Oversampling

In [48]:
# oversampling to acheive balance
from imblearn.over_sampling import RandomOverSampler
ros=RandomOverSampler(random_state=1)
over_sample_pipe=make_pipeline(preprocessor,ros,lr)
x_train_ros,y_train_ros=ros.fit_resample(x_train,y_train)
y_train_ros.value_counts()

,count
stroke,
0,750
1,750


In [62]:
# Build The model after oversampling with best parameter
lr=LogisticRegression(random_state=1, penalty='l2',solver="newton-cg",C=0.01)
over_sample_pipe=make_pipeline(preprocessor,ros,lr)
over_sample_pipe.fit(x_train_ros,y_train_ros)
y_pred=over_sample_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",over_sample_pipe.score(x_train_ros,y_train_ros))
print("accuracy Testing:",over_sample_pipe.score(x_test,y_test))




[[190  60]
 [ 16  19]]
              precision    recall  f1-score   support

           0       0.92      0.76      0.83       250
           1       0.24      0.54      0.33        35

    accuracy                           0.73       285
   macro avg       0.58      0.65      0.58       285
weighted avg       0.84      0.73      0.77       285

accuracy Training: 0.724
accuracy Testing: 0.7333333333333333


In [63]:
lr=LogisticRegression(random_state=1)
over_sample_pipe=make_pipeline(preprocessor,ros,lr)
over_sample_pipe.fit(x_train_ros,y_train_ros)
y_pred=over_sample_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",over_sample_pipe.score(x_train_ros,y_train_ros))
print("accuracy Testing:",over_sample_pipe.score(x_test,y_test))



[[195  55]
 [ 14  21]]
              precision    recall  f1-score   support

           0       0.93      0.78      0.85       250
           1       0.28      0.60      0.38        35

    accuracy                           0.76       285
   macro avg       0.60      0.69      0.61       285
weighted avg       0.85      0.76      0.79       285

accuracy Training: 0.8186666666666667
accuracy Testing: 0.7578947368421053


## Smote

In [68]:
#Create model pipeline with scaler, SMOTE, and model
from imblearn.over_sampling import SMOTE
smote = SMOTE()
log_reg_smote = LogisticRegression(random_state=1)
log_reg_smote_pipe = make_pipeline(preprocessor, smote, log_reg_smote)
#Fit and evaluate the model pipeline
log_reg_smote_pipe.fit(x_train, y_train)


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['id', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'age', 'ever_married', 'work_type', 'Residence_type',
       'smoking_status'],
      dtype='object'))])),
                ('smote', SMOTE()),
                ('logisticregression', LogisticRegression(random_state=1))])

In [69]:
# predict uing smote
y_pred=log_reg_smote_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",log_reg_smote_pipe.score(x_train,y_train))
print("accuracy Testing:",log_reg_smote_pipe.score(x_test,y_test))

[[199  51]
 [ 15  20]]
              precision    recall  f1-score   support

           0       0.93      0.80      0.86       250
           1       0.28      0.57      0.38        35

    accuracy                           0.77       285
   macro avg       0.61      0.68      0.62       285
weighted avg       0.85      0.77      0.80       285

accuracy Training: 0.7852112676056338
accuracy Testing: 0.7684210526315789


In [74]:
log_reg_smote = LogisticRegression(penalty='l2',solver="newton-cg",C=0.01,random_state=1)

log_reg_smote_pipe = make_pipeline(preprocessor, smote, log_reg_smote)
#Fit and evaluate the model pipeline
log_reg_smote_pipe.fit(x_train, y_train)
y_pred=log_reg_smote_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",log_reg_smote_pipe.score(x_train,y_train))
print("accuracy Testing:",log_reg_smote_pipe.score(x_test,y_test))

[[192  58]
 [ 16  19]]
              precision    recall  f1-score   support

           0       0.92      0.77      0.84       250
           1       0.25      0.54      0.34        35

    accuracy                           0.74       285
   macro avg       0.58      0.66      0.59       285
weighted avg       0.84      0.74      0.78       285

accuracy Training: 0.7511737089201878
accuracy Testing: 0.7403508771929824


**The Best logistic Regression Model was defult parameters with Smote with the less overfitting and balanced classes and the F1-score was 77**

# feature importances

In [44]:
# Obtain feature importances from the fit model
lr. feature_importances_

AttributeError: 'LogisticRegression' object has no attribute 'feature_importances_'

In [ ]:
# Saving the feature importances
feature_names=lr. feature_importances_
importances = pd. Series(lr. feature_importances_, index= feature_names,
name='Feature Importance' )

importances

In [ ]:
# Saving the feature importances sorted from smallest to largest (ascending=True)
sorted_importance = importances.sort_values()
sorted_importance

In [ ]:
# Using plt.gcf to get the fig
ax = sorted_importance.tail(10).plot(kind='barh',
figsize=(8,6), xlabel='Importance',
ylabel='Feature Name',
title='Top 10 Most Important Features')

fig_gcf = plt.gcf()

In [ ]:
def get_importances(model, feature_names=None, name='Feature Importance',
sort=False, ascending=True) :

## checking for feature names
  if feature_names == None:
      feature_names = model. feature_names_in_

## Saving the feature importances
  importances = pd. Series(model. feature_importances_, index= feature_names,name=name)

# sort importances
  if sort == True:
    importances = importances. sort_values(ascending=ascending)

   return importances

# CatBoost

In [46]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [126]:
# import Catbosstclassifier
from catboost import CatBoostClassifier
# build catboostmodel

cbc=CatBoostClassifier(random_state=1)
pipe=make_pipeline(preprocessor,cbc)
pipe.fit(x_train,y_train)
#evaluate
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))

Learning rate set to 0.009621
0:	learn: 0.6839715	total: 3.81ms	remaining: 3.81s
1:	learn: 0.6753530	total: 5.67ms	remaining: 2.83s
2:	learn: 0.6677480	total: 7.34ms	remaining: 2.44s
3:	learn: 0.6590313	total: 9.09ms	remaining: 2.26s
4:	learn: 0.6514893	total: 10.9ms	remaining: 2.17s
5:	learn: 0.6437210	total: 12.7ms	remaining: 2.1s
6:	learn: 0.6361502	total: 14.5ms	remaining: 2.06s
7:	learn: 0.6292995	total: 16.3ms	remaining: 2.02s
8:	learn: 0.6231752	total: 18.1ms	remaining: 1.99s
9:	learn: 0.6157800	total: 19.9ms	remaining: 1.97s
10:	learn: 0.6090417	total: 21.6ms	remaining: 1.95s
11:	learn: 0.6027175	total: 23.4ms	remaining: 1.93s
12:	learn: 0.5967388	total: 25.1ms	remaining: 1.91s
13:	learn: 0.5907185	total: 26.9ms	remaining: 1.9s
14:	learn: 0.5839684	total: 28.7ms	remaining: 1.88s
15:	learn: 0.5782013	total: 30.5ms	remaining: 1.88s
16:	learn: 0.5728200	total: 32.3ms	remaining: 1.87s
17:	learn: 0.5674006	total: 34.1ms	remaining: 1.86s
18:	learn: 0.5623280	total: 35.8ms	remaining: 

# Tunnung CatBoost

In [127]:
# Tunning catboost model
pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('num',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     StandardScaler())]),
                                    Index(['id', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'], dtype='object')),
                                   ('cat',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehotencoder',
                                                     OneHotEncoder(handle_unknown='ignore'))]),
                                    Index(['gender', 'age', 'ever_married', 'work_type', 'Residence_t

In [73]:

# build catboostmodel
cbc = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=2,
    loss_function='Logloss',
    auto_class_weights='Balanced',
    eval_metric='AUC',
    random_seed=1,
    verbose=200
)

pipe=make_pipeline(preprocessor,cbc)

pipe.fit(x_train,y_train)
#evaluate
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))

0:	total: 2.58ms	remaining: 2.58s
200:	total: 401ms	remaining: 1.59s
400:	total: 769ms	remaining: 1.15s
600:	total: 1.14s	remaining: 758ms
800:	total: 1.53s	remaining: 381ms
999:	total: 1.9s	remaining: 0us
[[240  10]
 [ 31   4]]
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       250
           1       0.29      0.11      0.16        35

    accuracy                           0.86       285
   macro avg       0.59      0.54      0.54       285
weighted avg       0.81      0.86      0.83       285

accuracy Training: 1.0
accuracy Testing: 0.856140350877193


# Smote with Catboost

In [76]:
smote = SMOTE()
cbc = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=2,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=1,
    verbose=200
)
cbc_smote_pipe = make_pipeline(preprocessor, smote,cbc)
#Fit and evaluate the model pipeline
cbc_smote_pipe.fit(x_train, y_train)
y_pred=cbc_smote_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",cbc_smote_pipe.score(x_train,y_train))
print("accuracy Testing:",cbc_smote_pipe.score(x_test,y_test))

0:	total: 18.2ms	remaining: 18.2s
200:	total: 1.95s	remaining: 7.74s
400:	total: 5.24s	remaining: 7.82s
600:	total: 7.41s	remaining: 4.92s
800:	total: 8.82s	remaining: 2.19s
999:	total: 11.2s	remaining: 0us
[[242   8]
 [ 29   6]]
              precision    recall  f1-score   support

           0       0.89      0.97      0.93       250
           1       0.43      0.17      0.24        35

    accuracy                           0.87       285
   macro avg       0.66      0.57      0.59       285
weighted avg       0.84      0.87      0.84       285

accuracy Training: 1.0
accuracy Testing: 0.8701754385964913


In [96]:
smote = SMOTE()
cbc = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=1050,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=1,
    verbose=200
)
cbc_smote_pipe = make_pipeline(preprocessor, smote,cbc)
#Fit and evaluate the model pipeline
cbc_smote_pipe.fit(x_train, y_train)
y_pred=cbc_smote_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",cbc_smote_pipe.score(x_train,y_train))
print("accuracy Testing:",cbc_smote_pipe.score(x_test,y_test))

0:	total: 16.5ms	remaining: 16.5s
200:	total: 2.9s	remaining: 11.5s
400:	total: 6.84s	remaining: 10.2s
600:	total: 10.7s	remaining: 7.11s
800:	total: 13.2s	remaining: 3.27s
999:	total: 14.2s	remaining: 0us
[[238  12]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.95      0.92       250
           1       0.20      0.09      0.12        35

    accuracy                           0.85       285
   macro avg       0.54      0.52      0.52       285
weighted avg       0.80      0.85      0.82       285

accuracy Training: 0.9143192488262911
accuracy Testing: 0.8456140350877193


In [138]:
smote = SMOTE()
cbc = CatBoostClassifier(verbose=200,
    l2_leaf_reg=200, eval_metric='AUC',)
cbc_smote_pipe = make_pipeline(preprocessor, smote,cbc)
#Fit and evaluate the model pipeline
cbc_smote_pipe.fit(x_train, y_train)
y_pred=cbc_smote_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",cbc_smote_pipe.score(x_train,y_train))
print("accuracy Testing:",cbc_smote_pipe.score(x_test,y_test))

0:	total: 20.5ms	remaining: 20.5s
200:	total: 2.73s	remaining: 10.9s
400:	total: 4.86s	remaining: 7.26s
600:	total: 7.69s	remaining: 5.11s
800:	total: 9.76s	remaining: 2.42s
999:	total: 11.6s	remaining: 0us
[[240  10]
 [ 29   6]]
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       250
           1       0.38      0.17      0.24        35

    accuracy                           0.86       285
   macro avg       0.63      0.57      0.58       285
weighted avg       0.83      0.86      0.84       285

accuracy Training: 0.9248826291079812
accuracy Testing: 0.8631578947368421


## smotw with cbc (defult parameter)

In [120]:
smote = SMOTE()
cbc = CatBoostClassifier(verbose=200)
cbc_smote_pipe = make_pipeline(preprocessor, smote,cbc)
#Fit and evaluate the model pipeline
cbc_smote_pipe.fit(x_train, y_train)
y_pred=cbc_smote_pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",cbc_smote_pipe.score(x_train,y_train))
print("accuracy Testing:",cbc_smote_pipe.score(x_test,y_test))

Learning rate set to 0.01225
0:	learn: 0.6857026	total: 19.6ms	remaining: 19.6s
200:	learn: 0.2643368	total: 2.6s	remaining: 10.3s
400:	learn: 0.1879282	total: 3.63s	remaining: 5.42s
600:	learn: 0.1455497	total: 4.66s	remaining: 3.1s
800:	learn: 0.1161794	total: 5.69s	remaining: 1.41s
999:	learn: 0.0937200	total: 6.7s	remaining: 0us
[[242   8]
 [ 30   5]]
              precision    recall  f1-score   support

           0       0.89      0.97      0.93       250
           1       0.38      0.14      0.21        35

    accuracy                           0.87       285
   macro avg       0.64      0.56      0.57       285
weighted avg       0.83      0.87      0.84       285

accuracy Training: 0.9812206572769953
accuracy Testing: 0.8666666666666667
